# 양식 신호 마스킹: 지웠을 때 무엇이 변하나

모델이 근거로 쓰는 상위 문구에 요구사항의 내용이 아니라 **작성 양식**을 가리키는 표현이
올라왔다. 그 표현을 규칙으로 지운 입력으로 같은 LODO를 다시 돌린 결과를 본다.

| 규칙 | 정의 |
|---|---|
| R1 | 요구를 *누가* 하는지 가리키는 말을 `<주체>`로 치환 |
| R2 | 의무·서술 어미를 어간으로 (`제시하여야` → `제시`) |
| R3 | 명사 뒤 조사 제거 (`데이터를`·`데이터의` → `데이터`) |

**점수를 올리려는 실험이 아니다.** 떨어지는 폭이 답이다 — 작으면 내용으로도 잡힌다는
뜻이고, 크면 새 발주처가 표기를 바꿀 때 무너진다는 뜻이다.

효과와 잡음은 평균 차이로 가르지 않는다. 문서가 10개뿐이라 fold 분산이 크므로
**10 fold 중 우세 fold 수**를 함께 보고, 사전에 **8/10**을 기준으로 정했다.

재현 명령:

```
$env:RFP_DATASET_VERSION='v4'; python -m scripts.evaluation.text_masking_ablation --masks subject ending josa subject+josa subject+ending+josa
```

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
sys.path.insert(0, str(ROOT))
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False

from scripts.data.preprocess_text import apply_mask

report = json.loads((ROOT / 'reports/current/v4/text_masking_ablation.json').read_text(encoding='utf-8'))
ARMS = ['subject', 'ending', 'josa', 'subject+josa', 'subject+ending+josa']
NAMES = {'subject': 'R1 주체', 'ending': 'R2 어미', 'josa': 'R3 조사', 'subject+josa': 'R1+R3', 'subject+ending+josa': 'R1+R2+R3'}
print(f"{len(report['specs'])}개 모델 x {len(ARMS)}개 규칙 + 기준선")

In [ ]:
sample = '제안사는 발주기관과 협의하여 데이터를 제공하여야 한다.'
display(pd.DataFrame(
    [{'규칙': '원문', '입력': sample}]
    + [{'규칙': NAMES[a], '입력': apply_mask(sample, a)} for a in ARMS]
).set_index('규칙').style.set_properties(**{'text-align': 'left'}))

In [ ]:
rows = []
for spec, arms in report['specs'].items():
    rows.append({'모델': spec, '입력': '원문', 'macro F1': arms['none']['macro_f1'], '차이': 0.0, '우세': None, '계약 recall': arms['none']['review_recall']})
    for arm in ARMS:
        c = arms[arm]['comparison']
        rows.append({'모델': spec, '입력': NAMES[arm], 'macro F1': arms[arm]['macro_f1'],
                     '차이': c['macro_f1']['difference'], '우세': c['fold_wins'],
                     '계약 recall': arms[arm]['review_recall']})
table = pd.DataFrame(rows)
display(table.pivot(index='입력', columns='모델', values='macro F1').reindex(['원문'] + [NAMES[a] for a in ARMS]).round(3))
display(table[table['입력'] != '원문'].pivot(index='입력', columns='모델', values='우세').reindex([NAMES[a] for a in ARMS]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pivot = table[table['입력'] != '원문'].pivot(index='입력', columns='모델', values='차이').reindex([NAMES[a] for a in ARMS])
pivot.plot.bar(ax=axes[0], rot=15)
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('원문 대비 macro F1 차이'); axes[0].set_ylabel('차이')

best = report['specs']['word 1-2 + char 3-4gram + balanced']
folds = pd.DataFrame({NAMES[a]: best[a]['comparison']['fold_macro_f1_difference'] for a in ARMS})
folds.plot.box(ax=axes[1], rot=15)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('word+char · fold별 차이 분포 (10 fold)')
plt.tight_layout()

recall = table[table['모델'] == 'word 1-2 + char 3-4gram + balanced'][['입력', '계약 recall']].set_index('입력')
display(recall.round(3))

## 읽는 법

- **단일 규칙 셋은 모두 잡음이다.** 최고가 6/10으로 사전 기준 8/10에 못 미친다. R2는
  word+char에서 +0.010인데 char·word 단독에서는 -0.010, -0.011로 **부호가 뒤집힌다.**
  효과라면 이렇게 갈리지 않는다.
- **R1이 특히 중요하다.** 924건 중 365건(39.5%)의 주체 표기를 지웠는데 macro F1이 어느
  모델에서도 내려가지 않았다. 사전에 "계약 recall이 떨어진다"고 적어둔 예측이 틀렸다 —
  char +0.023, word +0.015로 올랐다. `제안사` lift 1.96은 진짜였지만 **중복된 신호**였고,
  같은 요구사항의 내용어가 이미 같은 일을 하고 있었다. 조항 문체 의존이 낮다는 뜻이다.
- **셋을 겹치면 달라진다. 다만 탐색이다.** R1+R3만으로는 이득이 없는데(-0.001) 셋을 모두
  켜면 세 모델이 같은 방향으로 오르고(+0.018 / +0.015 / +0.011) 계약 recall도 0.568 →
  0.602가 된다. 그래도 채택하지 않는다 — 7/10으로 기준 미달이고, **단일 결과를 보고
  조합을 고른 사후 선택**이며, R2가 단독으로는 손해인데 조합에서만 이득이 되는 이유를
  설명하지 못한다.

같은 924건에서 다시 재는 것으로는 확인되지 않는다. 그 데이터를 보고 고른 조합이기
때문이다. 확인 조건은 새 RFP이거나 **다른 모델 계열**이며, 인코더 파인튜닝 비교에서
원문 대 R1+R2+R3을 사전 등록해 두면 후자가 성립한다.